# Claude로 moderation filter 구축
이 가이드는 사용자 생성 텍스트를 위한 content moderation filter를 Claude로 구축하는 방법을 보여 줍니다. 핵심 아이디어는 moderation 규칙과 category를 프롬프트에 직접 정의하여 쉽게 사용자 지정하고 실험할 수 있게 하는 것입니다.

## 기본 접근 방식
기본 접근 방식은 필터링할 category(예: "ALLOW"와 "BLOCK")를 설명하는 프롬프트와, 각 category에 어떤 콘텐츠가 포함되어야 하는지에 대한 상세 설명이나 예제를 Claude에 제공하는 것입니다. 그런 다음 분류할 사용자 생성 텍스트를 프롬프트의 일부로 넣고, 제공된 지침을 바탕으로 분류하도록 Claude에 요청합니다.

프롬프트 구조 예제입니다.

```text
You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- [Description or examples of content that should be blocked]

ALLOW CATEGORY:
- [Description or examples of content that is allowed]

Here is the user-generated text to categorize:
<user_text>{{USER_TEXT}}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
```

이를 사용하려면 `{{USER_TEXT}}`를 실제 분류할 사용자 생성 텍스트로 바꾸고 Claude API로 프롬프트를 보냅니다. Claude의 응답은 "ALLOW" 또는 "BLOCK"이어야 하며, 제공한 지침에 따라 텍스트를 어떻게 처리할지 나타냅니다.

## 사용 예제
이 접근 방식을 사용하는 방법을 보여 주는 Python 코드 예제입니다.

In [ ]:
%pip install anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"


def moderate_text(user_text, guidelines):
    prompt_template = """
    You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

    {guidelines}

    Here is the user-generated text to categorize:
    <user_text>{user_text}</user_text>

    Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
    """

    # user text로 프롬프트 형식 지정
    prompt = prompt_template.format(user_text=user_text, guidelines=guidelines)

    # Claude에 프롬프트를 보내고 응답 받기
    response = (
        client.messages.create(
            model=MODEL_NAME, max_tokens=10, messages=[{"role": "user", "content": prompt}]
        )
        .content[0]
        .text
    )

    return response

이 함수로 사용자 댓글 배열을 moderate하는 방법의 예제입니다.

In [3]:
example_guidelines = """BLOCK CATEGORY:
    - Promoting violence, illegal activities, or hate speech
    - Explicit sexual content
    - Harmful misinformation or conspiracy theories

    ALLOW CATEGORY:
    - Most other content is allowed, as long as it is not explicitly disallowed
"""

user_comments = [
    "This movie was great, I really enjoyed it. The main actor really killed it!",
    "Delete this post now or you better hide. I am coming after you and your family.",
    "Stay away from the 5G cellphones!! They are using 5G to control you.",
    "Thanks for the helpful information!",
]

for comment in user_comments:
    classification = moderate_text(comment, example_guidelines)
    print(f"Comment: {comment}\nClassification: {classification}\n")

Comment: This movie was great, I really enjoyed it. The main actor really killed it!
Classification: ALLOW

Comment: Delete this post now or you better hide. I am coming after you and your family.
Classification: BLOCK

Comment: Stay away from the 5G cellphones!! They are using 5G to control you.
Classification: BLOCK

Comment: Thanks for the helpful information!
Classification: ALLOW



## 사용자 지정

이 접근 방식의 핵심 이점 중 하나는 프롬프트의 "BLOCK"과 "ALLOW" category에 제공한 설명이나 예제를 수정하여 moderation 규칙을 쉽게 사용자 지정할 수 있다는 것입니다. 이를 통해 자신의 구체적인 요구나 선호에 맞춰 filtering을 조정할 수 있습니다.

예를 들어 Claude가 롤러코스터 애호가 포럼을 moderate하여 게시물이 주제를 벗어나지 않게 하려면, "ALLOW"와 "BLOCK" category 설명을 그에 맞게 갱신할 수 있습니다.

In [4]:
rollercoaster_guidelines = """BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals
"""

post_titles = [
    "Top 10 Wildest Inversions on Steel Coasters",
    "My Review of the New RMC Raptor Coaster at Cedar Point",
    "Best Places to Buy Cheap Hiking Gear",
    "Rumor: Is Six Flags Planning a Giga Coaster for 2025?",
    "My Thoughts on the Latest Marvel Movie",
]

for title in post_titles:
    classification = moderate_text(title, rollercoaster_guidelines)
    print(f"Title: {title}\nClassification: {classification}\n")

Title: Top 10 Wildest Inversions on Steel Coasters
Classification: ALLOW

Title: My Review of the New RMC Raptor Coaster at Cedar Point
Classification: ALLOW

Title: Best Places to Buy Cheap Hiking Gear
Classification: BLOCK

Title: Rumor: Is Six Flags Planning a Giga Coaster for 2025?
Classification: ALLOW

Title: My Thoughts on the Latest Marvel Movie
Classification: BLOCK



## Chain of Thought(CoT)로 성능 개선

Claude의 content moderation 능력을 높일 수 있는 기법 하나는 "chain-of-thought"(CoT) prompting입니다. 이 접근 방식은 Claude가 최종 출력만 제공하는 대신 reasoning 과정을 단계적인 생각의 연결로 나누도록 장려합니다.

moderation에 chain of thought를 활용하려면 Claude가 `<thinking>` tag 안에서 과정을 명확한 단계로 나누도록 프롬프트에 명시적으로 지시할 수 있습니다. 예제는 다음과 같습니다.

In [8]:
cot_prompt = """You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

First, inside of <thinking> tags, identify any potentially concerning aspects of the post based on the guidelines below and consider whether those aspects are serious enough to block the post or not. Finally, classify this text as either ALLOW or BLOCK inside <output> tags. Return nothing else.

Given those instructions, here is the post to categorize:

<user_post>{user_post}</user_post>"""

user_post = "Introducing my new band - Coaster Shredders. Check us out on YouTube!!"

response = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": cot_prompt.format(user_post=user_post)}],
    )
    .content[0]
    .text
)

print(response)

<thinking>
The post appears to be promoting a band rather than discussing rollercoasters, theme parks, or the amusement industry. This falls under the "spam, advertisements, or self-promotion" category, which is grounds for blocking the post.
</thinking>

<output>BLOCK</output>


## 예제로 성능 개선
성능을 개선하는 다른 기법은 프롬프트에 몇 가지 예제를 추가하여 Claude가 원하는 분류를 더 잘 이해할 초기 training data 또는 "few-shot learning"을 제공하는 것입니다. 이는 텍스트 설명만으로 category 경계가 완전히 명확하지 않을 수 있는 미묘하거나 모호한 사례에 특히 도움이 됩니다. 예제를 포함하도록 프롬프트 template을 수정하는 예제는 다음과 같습니다.

In [9]:
examples_prompt = """You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

Here are some examples:
<examples>
Text: I'm selling weight loss products, check my link to buy!
Category: BLOCK

Text: I hate my local park, the operations and customer service are terrible. I wish that place would just burn down.
Category: BLOCK

Text: Did anyone ride the new RMC raptor Trek Plummet 2 yet? I've heard it's insane!
Category: ALLOW

Text: Hercs > B&Ms. That's just facts, no cap! Arrow > Intamin for classic woodies too.
Category: ALLOW
</examples>

Given those examples, here is the user-generated text to categorize:
<user_text>{user_text}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else."""

user_post = "Why Boomerang Coasters Ain't It (Don't @ Me)"

response = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": examples_prompt.format(user_text=user_post)}],
    )
    .content[0]
    .text
)

print(response)

ALLOW
